# Project 2: Exploratory Data Analysis (EDA)
## Dataset: Cleaned Orders Dataset
### Goal: Analyze patterns, trends, and distributions in the order records

In [2]:
import pandas as pd

orders = pd.read_excel("Cleaned_Orders_Dataset.xlsx")
orders.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


## Phase 1: Descriptive Statistics
Here we calculate the basic summary statistics for our numeric columns.
This gives us a feel for the "center" and "spread" of the data before we dig deeper.

In [3]:
numeric_cols = ['Quantity', 'UnitPrice', 'ItemsInCart', 'TotalPrice']

summary = orders[numeric_cols].describe()
print(summary)

          Quantity    UnitPrice  ItemsInCart   TotalPrice
count  1200.000000  1200.000000  1200.000000  1200.000000
mean      2.945833   356.412750     5.485000  1053.968300
std       1.407557   197.177146     2.281983   819.856558
min       1.000000    11.390000     1.000000    11.390000
25%       2.000000   186.062500     4.000000   410.520000
50%       3.000000   364.210000     5.000000   823.615000
75%       4.000000   521.570000     7.000000  1578.475000
max       5.000000   699.930000    10.000000  3456.400000


### Key Observations from Descriptive Statistics
- Average order quantity is 3 items, with a max of 5
- TotalPrice shows a gap between mean ($1,053) and median ($823), suggesting right skew driven by high-value orders
- UnitPrice ranges widely from $11 to $699, indicating a broad product catalog

## Phase 2: Outlier Detection
We use the IQR method to identify unusually high or low order values.
IQR (Interquartile Range) is the spread of the middle 50% of the data.
Any value that falls too far outside this range is flagged as an outlier.
Formula: Lower Bound = Q1 - 1.5 * IQR | Upper Bound = Q3 + 1.5 * IQR

In [4]:
Q1 = orders['TotalPrice'].quantile(0.25)
Q3 = orders['TotalPrice'].quantile(0.75)
price_iqr = Q3 - Q1

lower_bound = Q1 - 1.5 * price_iqr
upper_bound = Q3 + 1.5 * price_iqr

price_outliers = orders[(orders['TotalPrice'] < lower_bound) | (orders['TotalPrice'] > upper_bound)]

print(f"Lower Bound: {lower_bound:.2f}")
print(f"Upper Bound: {upper_bound:.2f}")
print(f"Number of outliers found: {len(price_outliers)}")
print(price_outliers[['OrderID', 'CustomerID', 'Product', 'Quantity', 'TotalPrice']])

Lower Bound: -1341.41
Upper Bound: 3330.41
Number of outliers found: 8
        OrderID CustomerID  Product  Quantity  TotalPrice
107   ORD200107     C16775  Printer         5     3353.75
326   ORD200326     C65986   Laptop         5     3352.40
328   ORD200328     C18404   Tablet         5     3370.20
469   ORD200469     C13877    Chair         5     3384.90
632   ORD200632     C67260   Laptop         5     3390.80
789   ORD200789     C57276   Tablet         5     3456.40
1065  ORD201065     C47778  Printer         5     3334.00
1122  ORD201122     C38840  Monitor         5     3390.95


### Outlier Analysis - TotalPrice
- IQR method flagged 8 orders above the upper bound of $3,330
- Lower bound came out negative (-$1,341), meaning no suspiciously low outliers exist
- All 8 outliers share the same pattern: Quantity = 5 (maximum) on high-value products
- These are not data errors, they are genuine high-value orders and worth noting as the top-spending segment

## Phase 3: Categorical Breakdown
Here we analyze the non-numeric columns to understand customer behavior patterns.
We look at product popularity, payment preferences, order status distribution, and referral sources.

In [5]:
print("=== Product Popularity ===")
print(orders['Product'].value_counts())

print("\n=== Payment Method Distribution ===")
print(orders['PaymentMethod'].value_counts())

print("\n=== Order Status Breakdown ===")
print(orders['OrderStatus'].value_counts())

print("\n=== Referral Source Breakdown ===")
print(orders['ReferralSource'].value_counts())

=== Product Popularity ===
Product
Printer    181
Tablet     179
Chair      178
Laptop     173
Desk       170
Monitor    163
Phone      156
Name: count, dtype: int64

=== Payment Method Distribution ===
PaymentMethod
Online         258
Cash           246
Credit Card    234
Debit Card     232
Gift Card      230
Name: count, dtype: int64

=== Order Status Breakdown ===
OrderStatus
Cancelled    250
Returned     247
Pending      237
Shipped      235
Delivered    231
Name: count, dtype: int64

=== Referral Source Breakdown ===
ReferralSource
Instagram    259
Email        250
Google       241
Facebook     228
Referral     222
Name: count, dtype: int64


### Key Observations from Categorical Breakdown
- Product demand is evenly spread, no single product dominates sales
- Customers use all payment methods almost equally, no strong preference
- Cancellations and Returns outnumber Delivered orders, this is a red flag worth investigating
- Instagram is the top referral source, followed closely by Email

## Phase 4: Trend Analysis
Here we analyze how orders and revenue change over time.
We convert the Date column into a proper datetime format, then group by month to spot seasonal patterns.

In [6]:
orders['Date'] = pd.to_datetime(orders['Date'])
orders['Month'] = orders['Date'].dt.to_period('M')

monthly_orders = orders.groupby('Month').agg(
    total_orders=('OrderID', 'count'),
    total_revenue=('TotalPrice', 'sum')
).reset_index()

print("=== Monthly Orders and Revenue ===")
print(monthly_orders.to_string())

=== Monthly Orders and Revenue ===
      Month  total_orders  total_revenue
0   2023-01            47       56685.75
1   2023-02            37       40117.66
2   2023-03            43       48609.37
3   2023-04            31       27751.71
4   2023-05            49       63836.84
5   2023-06            45       49500.19
6   2023-07            44       42820.66
7   2023-08            51       54352.14
8   2023-09            29       29526.67
9   2023-10            47       52607.85
10  2023-11            41       43079.67
11  2023-12            46       43754.73
12  2024-01            32       38528.08
13  2024-02            32       36909.57
14  2024-03            36       36030.90
15  2024-04            50       49613.14
16  2024-05            34       27909.11
17  2024-06            53       68068.54
18  2024-07            43       42963.98
19  2024-08            28       31991.07
20  2024-09            44       39794.98
21  2024-10            31       37226.97
22  2024-11           

### Key Observations from Trend Analysis
- Overall order volume and revenue show a gradual decline from 2023 to early 2025
- June 2024 was the peak month with 53 orders and $68,068 in revenue
- August/September and January consistently show dips across years, suggesting seasonal slowdowns
- Mid-2025 shows signs of recovery with June 2025 rebounding to 49 orders and $53,047

## Phase 5: Correlation Analysis
Here we examine relationships between numeric variables.
A correlation score ranges from -1 to +1.
Close to +1 means the two variables rise together, close to -1 means one rises as the other falls, close to 0 means no relationship.

In [7]:
numeric_cols = ['Quantity', 'UnitPrice', 'ItemsInCart', 'TotalPrice']

correlation_matrix = orders[numeric_cols].corr()
print("=== Correlation Matrix ===")
print(correlation_matrix.round(2))

=== Correlation Matrix ===
             Quantity  UnitPrice  ItemsInCart  TotalPrice
Quantity         1.00       0.01         0.65        0.62
UnitPrice        0.01       1.00         0.00        0.72
ItemsInCart      0.65       0.00         1.00        0.39
TotalPrice       0.62       0.72         0.39        1.00


### Key Observations from Correlation Analysis
- UnitPrice is the strongest predictor of TotalPrice (0.72), meaning product pricing drives revenue more than order volume
- Quantity and TotalPrice have a moderate relationship (0.62), confirming bulk orders push spending up
- Quantity and ItemsInCart are moderately linked (0.65), bigger cart shoppers tend to order more
- UnitPrice has no relationship with Quantity or ItemsInCart, price point doesn't affect how much people add to cart

## Executive Summary

### Problem Statement
This analysis explores 1,200 order records spanning January 2023 to June 2025 to uncover patterns in customer behavior, product performance, revenue trends, and operational health.

### Methodology
The dataset was first cleaned in Project 1 to handle missing values, duplicates, and formatting issues. In Project 2, exploratory data analysis was performed covering descriptive statistics, outlier detection, categorical breakdowns, time-series trend analysis, and correlation analysis.

### Key Findings

1. **Revenue is driven by product price, not volume.** UnitPrice has the strongest correlation with TotalPrice (0.72), meaning stocking and promoting higher-value products has more impact on revenue than pushing customers to order more quantity.

2. **Cancellations and Returns are a serious problem.** These two statuses combined account for 497 out of 1,200 orders, outnumbering Delivered orders (231). Nearly half the orders never reach the customer successfully.

3. **Order volume is gradually declining.** 2023 was the strongest year, with 2024 and early 2025 showing a downward trend. June 2024 was the peak month at $68,068 in revenue.

4. **High-value outliers are legitimate bulk buyers.** The 8 flagged outliers all involve maximum quantity orders on expensive products. These are the top-spending customers and deserve retention attention.

5. **Instagram is the leading acquisition channel.** At 259 referrals it edges out Email (250) and Google (241), though all channels perform closely.

### Recommendations
- Investigate the high cancellation and return rate as a priority, this is the biggest operational risk in the data
- Focus marketing on high unit-price products since they drive revenue more than volume
- Design retention strategies for the bulk-buying customer segment identified in the outlier analysis
- Monitor the declining order trend and investigate whether seasonal dips in August/September and January can be addressed with targeted promotions